In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

print("✅ Step 1: Imports loaded successfully (pyspark.sql.functions, DeltaTable)")

In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("✅ Step 2: Widget parameters retrieved")
print(f"   📌 table_name       = '{table_name}'")
print(f"   📌 landing_timestamp = '{landing_timestamp}'")
print(f"   📌 load_type         = '{load_type}'")
print(f"   📌 merge_keys        = '{merge_keys}'")

In [0]:
source_table = f"adb_databeli.bronze.{table_name}"
filter_expr = f"landing_timestamp='{landing_timestamp}'"

print(f"✅ Step 3: Reading from bronze layer")
print(f"   📖 Source table  : {source_table}")
print(f"   🔍 Filter applied: {filter_expr}")

bronze_df = spark.read.table(source_table).filter(filter_expr)
bronze_count = bronze_df.count()

print(f"   📊 Records read from bronze: {bronze_count}")
print(f"   📋 Schema: {', '.join([f'{c.name} ({c.dataType})' for c in bronze_df.schema.fields])}")


In [0]:
updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("updated_timestamp", current_timestamp())

target_table = f"adb_databeli.silver.{table_name}"

print(f"✅ Step 4: Preparing silver layer write")
print(f"   🎯 Target table: {target_table}")
print(f"   ➕ Added columns: insert_timestamp, updated_timestamp")
print(f"   🔄 Load type: {load_type}")
print("   " + "-" * 50)

if load_type == "FULL":
    print(f"   📝 Executing FULL overwrite:")
    print(f"      Query: updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"   ✅ FULL overwrite completed successfully.")

elif load_type == "APPEND":
    print(f"   📝 Executing APPEND:")
    print(f"      Query: updated_df.write.mode('append').saveAsTable('{target_table}')")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"   ✅ APPEND completed successfully.")

elif load_type == "MERGE":
    print(f"   📝 Executing MERGE:")
    print(f"      Merge keys: {merge_keys}")
    # If the target table doesn't exist yet, create it with the first load
    if not spark.catalog.tableExists(target_table):
        print(f"      ⚠️ Target table '{target_table}' does not exist. Creating with initial overwrite.")
        print(f"      Query: updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"   ✅ Initial table created successfully.")
    else:
        print(f"      ✅ Target table '{target_table}' exists. Proceeding with MERGE.")
        delta_table = DeltaTable.forName(spark, target_table)
        keys = [k.strip() for k in merge_keys.split(",")]
        merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        print(f"      Merge condition: {merge_condition}")

        # On update: refresh updated_timestamp but preserve original insert_timestamp
        update_set = {c: col(f"source.{c}") for c in updated_df.columns if c != "insert_timestamp"}
        print(f"      Update set (columns updated on match): {[c for c in updated_df.columns if c != 'insert_timestamp']}")
        print(f"      ℹ️ 'insert_timestamp' preserved on matched rows (not overwritten).")
        print(f"      On not matched: INSERT ALL columns")

        delta_table.alias("target").merge(
            updated_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(
            set=update_set
        ).whenNotMatchedInsertAll(
        ).execute()
        print(f"   ✅ MERGE completed successfully.")

if not load_type or load_type not in ['FULL', 'APPEND', 'MERGE']:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected FULL, APPEND, or MERGE.")

In [0]:
record_count = updated_df.count()

print(f"✅ Step 5: Record count")
print(f"   📊 Total records processed: {record_count}")

In [0]:
print(f"✅ Step 6: Exiting notebook with record_count = {record_count}")
print("   🏁 bronze_to_silver pipeline completed successfully!")
dbutils.notebook.exit(str(record_count))